# Primera entrega: Sistema experto para determinar el perfil del inversionista

---

## Objetivo

Construir un sistema experto basado en reglas que clasifique a un cliente en uno de los perfiles: **Adverso al Riesgo**, **Moderado**, **Propenso al Riesgo** o **Sofisticado**.

Esta primera entrega contiene los tres componentes solicitados:

1. **Base de conocimientos:** preguntas, opciones, reglas de puntuación y perfiles.
2. **Motor de inferencia:** validación, regla de excepción, sumatoria y clasificación.
3. **Interfaz:** cuestionario interactivo para capturar respuestas y mostrar el dictamen.

## Marco teórico

### ¿Qué es un sistema experto?

Un sistema experto es un programa que intenta resolver un problema de una forma parecida a como lo haría una persona con experiencia en un tema. No adivina ni aprende por su cuenta: se le proporcionan reglas claras y, con base en ellas, llega a una conclusión.

En este caso, el papel del experto lo toma un asesor financiero. Sus criterios se convierten en preguntas y reglas para identificar qué tanto riesgo puede asumir una persona al invertir.

### Componentes principales

- **Base de conocimientos:** es como el manual del asesor. Aquí se guardan las preguntas, las opciones, los puntos y los posibles perfiles.
- **Motor de inferencia:** es la parte que consulta ese manual y toma la decisión. Revisa las respuestas, aplica las reglas y determina el perfil.
- **Interfaz de usuario:** es la parte que ve y utiliza la persona. Sirve para responder el cuestionario y recibir el resultado.

### ¿Cómo funciona este caso?

Primero, la persona responde el cuestionario. Después, el sistema revisa si cumple la condición especial para ser inversionista sofisticado. Si no la cumple, suma los puntos de las respuestas y compara el total con los rangos establecidos. Finalmente, muestra el perfil asignado y explica de manera breve por qué llegó a esa conclusión.

## 1. Base de conocimientos

En la práctica, esta parte es la memoria del sistema. Aquí queda escrito todo lo que el sistema necesita saber antes de atender a una persona: qué preguntar, qué significa cada respuesta y qué resultado puede dar.

Por ejemplo, una respuesta que muestra mayor experiencia o disposición al riesgo obtiene más puntos. En cambio, una respuesta que indica necesidad de seguridad o liquidez obtiene menos. La pregunta 12 se trata de forma especial porque identifica a una persona que ya cumple requisitos para considerarse inversionista sofisticado.

### Preguntas y reglas de puntuación

El siguiente bloque guarda las 14 preguntas del cuestionario, sus posibles respuestas y los puntos que recibe cada respuesta. Estos datos son la información que el sistema consulta para tomar una decisión.

### Datos del cuestionario

El siguiente bloque reúne las herramientas básicas que usará el cuaderno y registra las preguntas, las respuestas disponibles y los puntos asignados a cada una.

In [ ]:
# Herramientas necesarias para organizar los datos y mostrar el cuestionario.
from dataclasses import dataclass, field
from typing import Dict, List, Tuple
from IPython.display import display
import ipywidgets as W

# =====================================================================
# CATÁLOGO DE PREGUNTAS
# Cada pregunta contiene su texto y las opciones que verá el usuario.
# =====================================================================
CATALOGO_PREGUNTAS: Dict[int, Tuple[str, Dict[str, str]]] = {
    1: ("¿Cuál es su rango de edad?", {
        "A": "66 años o más",
        "B": "De 51 a 65 años",
        "C": "De 36 a 50 años",
        "D": "De 18 a 35 años",
    }),
    2: ("¿En cuánto tiempo estima que necesitará disponer del dinero invertido?", {
        "A": "En más de 5 años",
        "B": "Entre 3 y 5 años",
        "C": "Entre 1 y 3 años",
        "D": "En menos de 1 año",
    }),
    3: ("¿Cuenta usted con otros ingresos además del capital que desea invertir?", {
        "A": "Sí, cuento con ingresos recurrentes adicionales",
        "B": "No, este capital representa mi fuente principal",
    }),
    4: ("¿Qué porcentaje de su patrimonio total representa esta inversión?", {
        "A": "Más del 60 %",
        "B": "Entre el 30 % y el 60 %",
        "C": "Menos del 30 %",
    }),
    5: ("¿Cuál es el objetivo principal de esta inversión?", {
        "A": "Hacer crecer mi capital de forma balanceada",
        "B": "Maximizar el rendimiento aunque implique mayor riesgo",
        "C": "Preservar el capital y mantener liquidez",
    }),
    6: ("¿Depende usted económicamente de los rendimientos de esta inversión?", {
        "A": "No dependo de estos rendimientos",
        "B": "Sí, dependo de ellos para mis gastos",
    }),
    7: ("¿En qué instrumentos ha invertido anteriormente?", {
        "A": "Instrumentos de deuda gubernamental y pagarés bancarios",
        "B": "No he invertido antes / sólo cuentas de ahorro",
        "C": "Deuda corporativa y fondos de inversión mixtos",
        "D": "Acciones e instrumentos de renta variable",
        "E": "Derivados, divisas o productos estructurados",
    }),
    8: ("Ante una caída del 20 % en el valor de su portafolio, usted:", {
        "A": "Liquidaría toda la posición de inmediato",
        "B": "Liquidaría una parte para limitar la pérdida",
        "C": "Mantendría la posición esperando la recuperación",
        "D": "Aprovecharía para comprar a precios más bajos",
    }),
    9: ("¿Qué nivel de pérdida estaría dispuesto a tolerar en un año?", {
        "A": "Ninguna pérdida",
        "B": "Hasta un 5 %",
        "C": "Hasta un 15 %",
        "D": "Más del 25 % si el potencial de ganancia lo justifica",
    }),
    10: ("¿Cómo describiría su conocimiento sobre los mercados financieros?", {
        "A": "Nulo",
        "B": "Básico",
        "C": "Intermedio",
        "D": "Avanzado",
    }),
    11: ("¿Con qué frecuencia da seguimiento a sus inversiones?", {
        "A": "Nunca o casi nunca",
        "B": "Una vez al año",
        "C": "Mensualmente",
        "D": "Semanalmente",
        "E": "Diariamente",
    }),
    12: ("¿Se considera usted un inversionista sofisticado conforme a la normatividad vigente?", {
        "A": "Sí",
        "B": "No",
    }),
    13: ("¿Estaría dispuesto a invertir en instrumentos que no garantizan el capital?", {
        "A": "Sí",
        "B": "No",
    }),
    14: ("¿Acepta invertir en instrumentos con baja liquidez o plazo forzoso?", {
        "A": "Sí",
        "B": "No",
    }),
}

# =====================================================================
# REGLAS DE PUNTUACIÓN
# Cada opción tiene puntos según el nivel de riesgo que representa.
# =====================================================================
REGLAS_PUNTUACION: Dict[int, Dict[str, int]] = {
    1:  {"A": 1, "B": 2, "C": 3, "D": 4},
    2:  {"A": 4, "B": 3, "C": 2, "D": 1},
    3:  {"A": 2, "B": 1},
    4:  {"A": 1, "B": 2, "C": 3},
    5:  {"A": 2, "B": 3, "C": 1},
    6:  {"A": 2, "B": 1},
    7:  {"A": 2, "B": 1, "C": 3, "D": 4, "E": 4},
    8:  {"A": 1, "B": 2, "C": 3, "D": 4},
    9:  {"A": 1, "B": 2, "C": 3, "D": 4},
    10: {"A": 1, "B": 2, "C": 3, "D": 4},
    11: {"A": 1, "B": 2, "C": 3, "D": 4, "E": 4},
    12: {"A": 2, "B": 1},
    13: {"A": 2, "B": 1},
    14: {"A": 2, "B": 1},
}

# Regla especial: esta respuesta asigna directamente el perfil sofisticado.
PREGUNTA_SOFISTICADO = 12
RESPUESTA_SOFISTICADO = "A"

print(f"Base de conocimientos cargada: {len(CATALOGO_PREGUNTAS)} preguntas y {len(REGLAS_PUNTUACION)} reglas de puntuación.")

### Perfiles posibles

El siguiente bloque define los cuatro resultados que puede mostrar el sistema. Los primeros tres dependen de la cantidad de puntos obtenida; el perfil sofisticado se asigna cuando se cumple la condición especial de la pregunta 12.

In [ ]:
@dataclass(frozen=True)
class Perfil:
    clave: str
    nombre: str
    descripcion: str
    minimo: int | None
    maximo: int | None

    def contiene(self, puntaje: int) -> bool:
        return self.minimo is not None and self.minimo <= puntaje <= self.maximo

PERFILES = {
    "adverso": Perfil("adverso", "Adverso al Riesgo", "Prioriza la preservación del capital y una baja exposición al riesgo.", 0, 14),
    "moderado": Perfil("moderado", "Moderado", "Busca equilibrio entre seguridad y rendimiento, aceptando cierto riesgo.", 15, 28),
    "propenso": Perfil("propenso", "Propenso al Riesgo", "Acepta una exposición alta al riesgo para buscar mayor rentabilidad.", 29, 64),
    "sofisticado": Perfil("sofisticado", "Sofisticado", "Acredita experiencia, patrimonio o ingresos para operar como inversionista sofisticado.", None, None),
}

for perfil in PERFILES.values():
    rango = "Regla de excepción: P12 = A" if perfil.minimo is None else f"{perfil.minimo} a {perfil.maximo} puntos"
    print(f"- {perfil.nombre}: {rango}")

## 2. Motor de inferencia

En palabras sencillas, esta es la parte que hace las cuentas y toma la decisión. Es como tener al asesor leyendo las respuestas una por una, siguiendo siempre el mismo criterio y sin dejarse llevar por opiniones personales.

Primero revisa que las respuestas sean válidas. Después verifica si aplica el caso especial de inversionista sofisticado; si no aplica, suma los puntos y compara el resultado con los rangos definidos.

También guarda una breve explicación de los pasos que siguió. Esto es importante porque no solo entrega una etiqueta: permite decir de dónde salió el resultado.

In [ ]:
@dataclass
class Resultado:
    perfil: Perfil
    puntaje: int
    regla: str
    traza: List[str] = field(default_factory=list)
    incidencias: List[str] = field(default_factory=list)

class SistemaExpertoFinanciero:
    def evaluar_perfil(self, respuestas: Dict[int, str]) -> Resultado:
        respuestas = {int(q): str(op).strip().upper() for q, op in respuestas.items()}
        traza, incidencias = [], []

        # R0: validación de hechos
        for q in CATALOGO_PREGUNTAS:
            if q not in respuestas:
                incidencias.append(f"P{q}: sin respuesta")
            elif respuestas[q] not in REGLAS_PUNTUACION[q]:
                incidencias.append(f"P{q}: opción inválida")
        traza.append(f"R0: se validaron {len(respuestas)} respuestas; incidencias: {len(incidencias)}.")

        # R1: regla de excepción con máxima prioridad
        if respuestas.get(PREGUNTA_SOFISTICADO) == RESPUESTA_SOFISTICADO:
            traza.append("R1: P12 = A; se concluye perfil SOFISTICADO sin realizar sumatoria.")
            return Resultado(PERFILES["sofisticado"], 0, "R1 - excepción", traza, incidencias)

        traza.append("R1: P12 no acredita perfil sofisticado; continúa la evaluación.")

        # R2: sumatoria ponderada
        puntaje = sum(REGLAS_PUNTUACION[q].get(respuestas.get(q), 0) for q in REGLAS_PUNTUACION)
        traza.append(f"R2: suma de los pesos de las respuestas = {puntaje}.")

        # R3: clasificación por rango
        for clave in ("adverso", "moderado", "propenso"):
            perfil = PERFILES[clave]
            if perfil.contiene(puntaje):
                traza.append(f"R3: {perfil.minimo} <= {puntaje} <= {perfil.maximo}; perfil {perfil.nombre.upper()}.")
                return Resultado(perfil, puntaje, "R3 - clasificación por rango", traza, incidencias)

        raise ValueError("El puntaje no está cubierto por los rangos de la base de conocimientos.")

sistema = SistemaExpertoFinanciero()
print("Motor de inferencia listo.")

## 3. Interfaz de usuario

Esta es la parte práctica que usaría la persona. En lugar de escribir datos dentro del programa, solo responde el cuestionario seleccionando opciones, igual que en un formulario.

La siguiente celda crea el cuestionario. Al pulsar **Evaluar perfil**, el sistema toma las respuestas, las manda al motor de inferencia y devuelve una respuesta fácil de entender: el perfil obtenido, una descripción y los pasos generales que siguió para decidirlo.

In [ ]:
controles = {}
bloques = []
for q, (enunciado, opciones) in CATALOGO_PREGUNTAS.items():
    titulo = W.HTML(f"<b>{q}. {enunciado}</b>")
    control = W.RadioButtons(options=[(f"{clave}) {texto}", clave) for clave, texto in opciones.items()], value=None)
    controles[q] = control
    bloques.extend([titulo, control])

boton = W.Button(description="Evaluar perfil", button_style="primary")
limpiar = W.Button(description="Limpiar")
salida = W.Output()

def evaluar(_):
    respuestas = {q: control.value for q, control in controles.items() if control.value is not None}
    with salida:
        salida.clear_output()
        if len(respuestas) != len(CATALOGO_PREGUNTAS):
            print("Completa las 14 preguntas antes de evaluar.")
            return
        resultado = sistema.evaluar_perfil(respuestas)
        print(f"Perfil asignado: {resultado.perfil.nombre}")
        print(f"Puntaje: {'No aplica' if resultado.regla.startswith('R1') else resultado.puntaje}")
        print(f"Regla final: {resultado.regla}")
        print(f"Descripción: {resultado.perfil.descripcion}\n")
        print("Traza de razonamiento:")
        for paso in resultado.traza:
            print("•", paso)

def reiniciar(_):
    for control in controles.values():
        control.value = None
    salida.clear_output()

boton.on_click(evaluar)
limpiar.on_click(reiniciar)
display(W.VBox([W.HTML("<h3>Cuestionario de perfil del inversionista</h3>"), *bloques, W.HBox([boton, limpiar]), salida], layout=W.Layout(max_width="820px")))

## Conclusión

El sistema implementa una base de conocimientos separada del algoritmo, un motor de inferencia explicable y una interfaz para registrar los hechos del caso. Esta separación permite modificar preguntas, pesos o rangos sin rehacer el flujo completo del sistema experto.